In [ ]:
!pip install transformers datasets torch seqeval evaluate numpy tokenizers


### Need to Add Custom Tokenizer Here

In [ ]:
# Load tokenizer
model_checkpoint = "distilbert-base-uncased"
# tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# Tokenize and align labels
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"], truncation=True, is_split_into_words=True, padding=True
    )
    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        aligned_labels = [-100 if word_id is None else label[word_id] for word_id in word_ids]
        labels.append(aligned_labels)
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Tokenize the dataset
tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=True)

# Ensure the train-test split works correctly
try:
    split = tokenized_dataset.train_test_split(test_size=0.2)
    train_dataset = split["train"]
    test_dataset = split["test"]
except Exception as e:
    print(f"Error splitting dataset: {e}")
    train_dataset, test_dataset = tokenized_dataset, tokenized_dataset  # Use all data if splitting fails



In [ ]:
# Initialize model
model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(label2id)
)
model.config.id2label = id2label
model.config.label2id = label2id

# Define training arguments
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    save_strategy="epoch",
    logging_dir="./logs"
)

# Data collator
data_collator = DataCollatorForTokenClassification(tokenizer)

# Evaluation metric
metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)
    true_predictions = [
        [id2label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [id2label[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

# Train the model
trainer.train()


In [ ]:
# Evaluate the model
results = trainer.evaluate()
print("Evaluation Results:", results)

In [ ]:
import torch
import json

# Test the model with a prediction function
def predict_ner(model, tokenizer, text):
    tokens = tokenizer(text, truncation=True, padding=True, return_tensors="pt", is_split_into_words=False)
    model.eval()
    with torch.no_grad():
        outputs = model(**tokens)
    logits = outputs.logits
    predictions = torch.argmax(logits, dim=2)
    input_ids = tokens["input_ids"].squeeze().tolist()
    decoded_tokens = tokenizer.convert_ids_to_tokens(input_ids)
    predicted_labels = [model.config.id2label[pred] for pred in predictions.squeeze().tolist()]
    return list(zip(decoded_tokens, predicted_labels))

example_text = """Aa
<o
Perfect Vision Opticals

28, 1st Floor, 2nd Cross, 1st Block, RT Nagar, Mumbai-400050
GSTIN: 28AJHCU1234L1ZA PAN: RFRBG5397G

INVOICE

Invoice No: 39787 BILL TO

Invoice Date: 25-11-2021 Deepa Patel

P.O.No: 3888 73, Pine Boulevard

P.O.Date: 23-11-2021 Mangalore

Ref No: 5984 Pan No: EXTQG2481B

Ref Date: 25-11-2021 Mobile No: 9855902017
Email: deepapatel@gmail.com
GSTIN: 24GRZOB1843N1Z5

Description In Detail Discount

Polarized Wrap Around Sports 3479.00 9915.15
Sunglasses ,

Ray-Ban Aviator Classic 2957.00 5618.30
Sunglasses

Bobster Eyewear Fuel
Photochromic Goggles 2684.00 2549.80

TOTAL 18083.25

Sub Total 18083.25
IGST @ 18% 3254.99

Grand Total 21338.24
PAYMENT METHOD
we Accept the PayPal, Master Card, VISA and E- Wallets
Terms & Condition:

1. Goods Once Sold Will Not be Taken Back
2. Goods Can be Exchanged During 2:00 pm to 4:00 pm

Thank you for business & visit Again



"""
predictions = predict_ner(model, tokenizer, example_text)

# Format the output in "label: token" format
formatted_output = []
for token, label in predictions:
    # if label != "O":  # Skip "O" tags (non-entities)
      formatted_output.append(f"{label}: {token}")

# Print the formatted output
for line in formatted_output:
    # write to a file
    with open('output.txt', 'a') as f:
        f.write(line + '\n')
    print(line)


In [ ]:
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification
import evaluate
import numpy as np

# Load the tagged dataset
def load_tagged_dataset(file_path):
    df = pd.read_csv(file_path)
    grouped = df.groupby("Filename")
    data = {
        "id": [],
        "tokens": [],
        "ner_tags": []
    }
    label2id = {}  # Mapping from tags to IDs
    id2label = {}  # Mapping from IDs to tags
    label_counter = 0

    for filename, group in grouped:
        tokens = group["Token"].tolist()
        tags = group["Tag"].tolist()

        # Map tags to numeric IDs
        numeric_tags = []
        for tag in tags:
            if tag not in label2id:
                label2id[tag] = label_counter
                id2label[label_counter] = tag
                label_counter += 1
            numeric_tags.append(label2id[tag])

        data["id"].append(filename)
        data["tokens"].append(tokens)
        data["ner_tags"].append(numeric_tags)

    return Dataset.from_dict(data), label2id, id2label

# Prepare dataset
file_path = './tagged_output.csv'  # Path to your tagged output CSV
dataset, label2id, id2label = load_tagged_dataset(file_path)

In [ ]:
import pandas as pd
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, processors

# Initialize a WordPiece tokenizer
tokenizer = Tokenizer(models.WordPiece(unk_token="[UNK]"))

# Define pre-tokenizer and special tokens
tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

# Set up the trainer
trainer = trainers.WordPieceTrainer(
    vocab_size=30000,  # Adjust based on your needs
    special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"]
)

# Load your dataset
# files = ["data.txt"]  # Path to your text files

# Load the tagged_output.csv
file_path = './tagged_output.csv'
data = pd.read_csv(file_path)

# Extract tokens from the 'Token' column
tokens = data['Token'].tolist()

chunk_size = 50  # Number of tokens per line
output_file = './tokens_for_training.txt'

with open(output_file, 'w') as f:
    for i in range(0, len(tokens), chunk_size):
        chunk = tokens[i:i + chunk_size]
        f.write(" ".join(chunk) + "\n")  # Join tokens in the chunk with a space

output_file

# Train the tokenizer
tokenizer.train([output_file], trainer)

# Save the tokenizer to disk
tokenizer.save("custom_tokenizer.json")
print("Tokenizer trained and saved as custom_tokenizer.json")


In [ ]:
from transformers import PreTrainedTokenizerFast

# Load the custom tokenizer
tokenizer = PreTrainedTokenizerFast(tokenizer_file="custom_tokenizer.json")
tokenizer.add_special_tokens({"cls_token": "[CLS]", "sep_token": "[SEP]"})

# Tokenize an example sentence
text = "Bobster Eyewear Fuel Photochromic Goggles"
tokens = tokenizer(text)
print(tokens)


In [ ]:
import torch
from transformers import DistilBertForTokenClassification, PreTrainedTokenizerFast

# Load the custom tokenizer
tokenizer = PreTrainedTokenizerFast(tokenizer_file="custom_tokenizer.json")

# Initialize a pre-trained DistilBERT model
model = DistilBertForTokenClassification.from_pretrained("distilbert-base-uncased", num_labels=10)
model.resize_token_embeddings(len(tokenizer))

# Tokenize the input
text = "Polarized Wrap Around Sports Sunglasses"
inputs = tokenizer(text, return_tensors="pt")

# Remove token_type_ids if present
if "token_type_ids" in inputs:
    del inputs["token_type_ids"]

# Feed the inputs into the model
outputs = model(**inputs)

# Print the outputs
print(outputs)


In [ ]:
def format_predictions(outputs, tokenizer, inputs):
    """
    Format the model's output predictions into a readable format.
    """
    # Extract logits and input IDs
    logits = outputs.logits
    input_ids = inputs["input_ids"].squeeze().tolist()

    # Decode tokens and find predicted labels
    decoded_tokens = tokenizer.convert_ids_to_tokens(input_ids)
    predictions = torch.argmax(logits, dim=2).squeeze().tolist()

    # Map predictions to labels
    labels = [model.config.id2label[pred] for pred in predictions]

    # Combine tokens with their labels
    formatted_predictions = [
        {"token": token, "label": label}
        for token, label in zip(decoded_tokens, labels)
    ]

    return formatted_predictions

In [ ]:
# Ensure the tokenizer has a pad_token
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '[PAD]'})

# Resize the model embeddings to accommodate the new pad_token
model.resize_token_embeddings(len(tokenizer))

example_text="""Aa
<o
Perfect Vision Opticals

28, 1st Floor, 2nd Cross, 1st Block, RT Nagar, Mumbai-400050
GSTIN: 28AJHCU1234L1ZA PAN: RFRBG5397G

INVOICE

Invoice No: 39787 BILL TO

Invoice Date: 25-11-2021 Deepa Patel

P.O.No: 3888 73, Pine Boulevard

P.O.Date: 23-11-2021 Mangalore

Ref No: 5984 Pan No: EXTQG2481B

Ref Date: 25-11-2021 Mobile No: 9855902017
Email: deepapatel@gmail.com
GSTIN: 24GRZOB1843N1Z5

Description In Detail Discount

Polarized Wrap Around Sports 3479.00 9915.15
Sunglasses ,

Ray-Ban Aviator Classic 2957.00 5618.30
Sunglasses

Bobster Eyewear Fuel
Photochromic Goggles 2684.00 2549.80

TOTAL 18083.25

Sub Total 18083.25
IGST @ 18% 3254.99

Grand Total 21338.24
PAYMENT METHOD
we Accept the PayPal, Master Card, VISA and E- Wallets
Terms & Condition:

1. Goods Once Sold Will Not be Taken Back
2. Goods Can be Exchanged During 2:00 pm to 4:00 pm

Thank you for business & visit Again
"""

# Tokenize the input text
inputs = tokenizer(
    example_text,
    truncation=True,
    padding=True,
    return_tensors="pt",
    max_length=512  # Set a reasonable maximum length
)

# Remove token_type_ids if present in inputs
if "token_type_ids" in inputs:
    del inputs["token_type_ids"]

# Pass the inputs to the model
outputs = model(**inputs)

# Process predictions
logits = outputs.logits
predictions = torch.argmax(logits, dim=2)

# Decode tokens and labels
decoded_tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"].squeeze().tolist())
predicted_labels = [model.config.id2label[pred] for pred in predictions.squeeze().tolist()]  # Removed .item()

# Print formatted predictions
for token, label in zip(decoded_tokens, predicted_labels):
    print(f"Token: {token}, Label: {label}")
